<a href="https://colab.research.google.com/github/Gayathri-achari/Projects/blob/Movie-recommended-system/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.neighbors import NearestNeighbors
from ast import literal_eval


In [2]:
movies_df = pd.read_csv("/content/movies.csv.zip")
credits_df = pd.read_csv("/content/movies.csv.zip")


In [3]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    elif isinstance(x, str):
        return str.lower(x.replace(" ", ""))
    return ""

features = ['cast_name', 'title', 'director_name', 'genre']
for feature in features:
    movies_df[feature] = movies_df[feature].apply(clean_data)


In [4]:
def create_soup(x):
    return ' '.join(x['title']) + ' ' + ' '.join(x['cast_name']) + ' ' + x['director_name'] + ' ' + ' '.join(x['genre'])

movies_df["soup"] = movies_df.apply(create_soup, axis=1)


In [5]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies_df["soup"])
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)


In [6]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = movies_df.index[movies_df['title'] == title][0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movies_indices = [ind[0] for ind in sim_scores]
    return movies_df['title'].iloc[movies_indices]


In [7]:
# Load the rating data
!ls /content/

# If ratings.csv.zip is listed, the file exists. If not, you need to upload it.
# You can upload files by clicking on the folder icon on the left sidebar,
# then the upload icon, and selecting your file.

rating_data = pd.read_csv("/content/movies.csv.zip")

user_item_matrix = rating_data.pivot(index='user_id', columns='movie_id', values='imbd_rating').fillna(0)
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10)
knn_model.fit(user_item_matrix)

movies.csv.zip	sample_data


NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=10)

In [8]:
def movie_recommender_engine(movie_name, matrix, cf_model, n_recs=10):
    movie_id = process.extractOne(movie_name, movie_names['title'])[2]
    distances, indices = cf_model.kneighbors(matrix[movie_id], n_neighbors=n_recs)
    movie_rec_ids = sorted(list(zip(indices.squeeze().tolist(), distances.squeeze().tolist())), key=lambda x: x[1])[:0:-1]
    cf_recs = [{'Title': movie_names['title'][i[0]], 'Distance': i[1]} for i in movie_rec_ids]
    return pd.DataFrame(cf_recs, index=range(1, n_recs))


In [9]:
# Install fuzzywuzzy if you haven't already
!pip install fuzzywuzzy
from fuzzywuzzy import process

def movie_recommender_engine(movie_name, matrix, cf_model, n_recs=10):
    best_match_tuple = process.extractOne(movie_name, movies_df['title'])
    if best_match_tuple:
        movie_id = best_match_tuple[2]
    else:
        print(f"Movie '{movie_name}' not found in the dataset.")
        return pd.DataFrame()

    if movie_id not in matrix.columns:
         print(f"Movie ID {movie_id} not found in the user-item matrix columns.")
         return pd.DataFrame()
    movie_ratings_for_recommendation = matrix[movie_id].values.reshape(1, -1)


    distances, indices = cf_model.kneighbors(movie_ratings_for_recommendation, n_neighbors=n_recs)

    # Flatten the indices and distances arrays
    indices = indices.flatten()
    distances = distances.flatten()
    movie_rec_ids = sorted(list(zip(indices.tolist(), distances.tolist())), key=lambda x: x[1])[1:] # Start from index 1

    cf_recs = []
    for i, dist in movie_rec_ids:
         movie_ids_in_matrix = matrix.columns.tolist()
         try:
             recommended_movie_id = movie_ids_in_matrix[i]
             recommended_movie_title = movies_df['title'].iloc[i]
             cf_recs.append({'Title': recommended_movie_title, 'Distance': dist})
         except IndexError:
             print(f"Warning: Index {i} from collaborative filtering neighbors is out of bounds for movies_df.")
             continue


    return pd.DataFrame(cf_recs, index=range(1, len(cf_recs) + 1))

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [10]:
# Assuming clean_data function is already defined and imported
def get_recommendations(title, cosine_sim=cosine_sim):
    # Clean the input title
    cleaned_title = clean_data(title)
    # Find the index of the cleaned title in the DataFrame
    if cleaned_title in movies_df['title'].values:
        idx = movies_df.index[movies_df['title'] == cleaned_title][0]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:11]
        movies_indices = [ind[0] for ind in sim_scores]
        return movies_df['title'].iloc[movies_indices]
    else:
        # Handle cases where the movie title is not found after cleaning
        print(f"Movie '{title}' not found in the dataset after cleaning.")
        return pd.Series() # Return an empty Series or handle as appropriate

# %%
print("Recommendations for The Dark Knight Rises:")
# Clean the input title when calling the function
print(get_recommendations("The Dark Knight Rises"))

print("Collaborative Recommendations for Batman:")
print(movie_recommender_engine("Batman", user_item_matrix, knn_model))


Recommendations for The Dark Knight Rises:
13                  inception
24               interstellar
40                theprestige
53                    memento
68         thedarkknightrises
126              batmanbegins
0      theshawshankredemption
1                thegodfather
3          thegodfatherpartii
4                  12angrymen
Name: title, dtype: object
Collaborative Recommendations for Batman:
Movie ID 95 not found in the user-item matrix columns.
Empty DataFrame
Columns: []
Index: []


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Example true vs predicted ratings
true_ratings = [4.0, 3.5, 5.0, 2.0, 4.5]
predicted_ratings = [3.8, 3.0, 4.7, 2.5, 4.2]

# Compute metrics
mae = mean_absolute_error(true_ratings, predicted_ratings)
mse = mean_squared_error(true_ratings, predicted_ratings)
rmse = np.sqrt(mse)
r2 = r2_score(true_ratings, predicted_ratings)

print("Collaborative Filtering Accuracy Measures:")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R² Score:", r2)


Collaborative Filtering Accuracy Measures:
MAE : 0.36
MSE : 0.14399999999999996
RMSE: 0.3794733192202055
R² Score: 0.8641509433962264
